<a href="https://colab.research.google.com/github/OrastaRakhmatullayeva/ML-projects/blob/main/smartphone_screen_time_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smartphone Screen Time Prediction

Kaggle Playground Series (S6E8) dataset asosida foydalanuvchining kunlik ekran vaqtini (`daily_screen_time_hours`) Linear Regression yordamida bashorat qilish.

## 1. Import & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import opendatasets as od
od.download('https://www.kaggle.com/competitions/playground-series-s6e8/data')

In [ ]:
df = pd.read_csv('/content/playground-series-s6e8/train.csv')
df.shape

In [ ]:
df.info()

## 2. EDA (Missing Values & Outliers)

### 2.1 Missing values

In [ ]:
miss = df.isnull().sum() / len(df) * 100
miss.sort_values(ascending=False).round(2)

Bir nechta ustunda (`social_media_hours`, `gaming_hours`, `weekend_screen_time` and others) 10-20% oralig'ida missing value bor. Bu darajadagi bo'shliq qatorlarni o'chirishga emas, to'ldirishga asos beradi.

### 2.2 Outlier ko'rinishi (boxplot)

In [ ]:
num = df.select_dtypes(include="number")

n_cols = 3
n_rows = -(-len(num.columns) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num.columns):
    sns.boxplot(y=num[col], ax=axes[i])
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Vizual tekshiruv shuni ko'rsatdi: aniq outlier faqat 4 ta ustunda bor : `social_media_hours`, `gaming_hours`, `work_study_hours`, `weekend_screen_time`.

## 3. Data Cleaning

### 3.1 delete unnecessary columns

In [ ]:
df = df.drop(columns=['id'])

### 3.2 Filling missing values

In [ ]:
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum().sum()

### 3.3 Cleaning outlier with IQR (clip)

In [ ]:
outlier_cols = ['social_media_hours', 'gaming_hours', 'work_study_hours', 'weekend_screen_time']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_outliers} ta outlier, chegara [{lower:.2f}, {upper:.2f}]")

    df[col] = df[col].clip(lower=lower, upper=upper)

In [ ]:
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(16, 4))
for i, col in enumerate(outlier_cols):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

## 4. Feature Selection / Correlation

`daily_screen_time_hours` bilan boshqa raqamli ustunlar orasidagi korrelyatsiyani tekshiramiz, bu qaysi feature'lar target bilan kuchliroq bog'liqligini ko'rsatadi.

In [ ]:
corr = df.corr(numeric_only=True)['daily_screen_time_hours'].sort_values(ascending=False)
corr

**Yakka feature bilan sinovlar (R²):**

| Feature | R² |
|---|---|
| `age` | ~0.00 |
| `social_media_hours` | 0.27 |
| `work_study_hours` | 0.23 |
| `gaming_hours`, `sleep_hours`, `app_opens_per_day` (birga) | 0.13 |

Bitta `social_media_hours` ustuni, uch ustun (`gaming_hours`, `sleep_hours`, `app_opens_per_day`) birgalikdan ham kuchliroq bashoratchi bo'lib chiqdi ,demak barcha ustun ham target bilan bir xil darajada bog'liq emas, korrelyatsiyaga qarab tanlash kerak.

## 5. Model Training & Evaluation

In [ ]:
feats = [
    "social_media_hours", "gaming_hours", "app_opens_per_day",
    "notifications_per_day", "weekend_screen_time"
]

X = df[feats]
y = df["daily_screen_time_hours"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print(dict(zip(feats, model.coef_.round(4))))
print("intercept =", round(model.intercept_, 4))

In [ ]:
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("R2 =", round(r2, 4))
print("MAE =", round(mae, 4))
print("RMSE =", round(rmse, 4))

## 6. Xulosa

`social_media_hours`, `gaming_hours`, `app_opens_per_day`, `notifications_per_day`, `weekend_screen_time` feature'lari bilan qurilgan model eng yaxshi natijani berdi:

- **R² = 0.58** — modeldagi o'zgaruvchanlikning 58%ini tushuntiradi
- **MAE = 1.26 soat** — bashorat o'rtacha 1.26 soat farq qiladi
- **RMSE = 1.61 soat**

**Kuzatuvlar:**
- `age` yakka o'zi deyarli hech qanday bashorat kuchiga ega emas (R² ≈ 0)
- `social_media_hours` eng kuchli yakka bashoratchi (R² = 0.27) , bu mantiqiy, chunki u umumiy screen time'ning tarkibiy qismi
- Bir nechta o'rtacha-kuchli feature'ni birlashtirish yakka feature'lardan sezilarli yaxshiroq natija beradi

**Keyingi qadamlar:**
- Chiziqli bo'lmagan bog'liqliklarni tutish uchun Random Forest yoki Gradient Boosting sinab ko'rish
- Qo'shimcha feature engineering (masalan, combined screen-time metrikalar)